# Hˢ Standards Edition — Higgins Decomposition on the Simplex

**Hˢ = R ∘ M ∘ E ∘ C ∘ T ∘ V ∘ S**

A complete, self-contained compositional inference notebook. Bring your CSV — the notebook does the rest.

**What this notebook does:**
1. Installs all required packages (numpy only)
2. Downloads the Hˢ pipeline from GitHub (if not already local)
3. Runs the full 12-step decomposition on your data
4. Generates diagnostic codes, structural modes, and investigation prompts
5. Produces fingerprints, amalgamation stability tests, and power maps
6. Saves results as JSON for comparison and archiving

**How to use at a conference:**
- Download this single notebook
- Open in JupyterLab (`jupyter lab`)
- Run All Cells
- When prompted, upload your CSV (columns = carriers, rows = observations)
- Read the results — the instrument reads, you decide

---

*Peter Higgins — Rogue Wave Audio — PeterHiggins@RogueWaveAudio.com*
*github.com/PeterHiggins19/higgins-decomposition*
*CoDaWork 2026 — Coimbra, Portugal*
*License: CC BY 4.0*

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1: INSTALL & SETUP — Run this first
# ════════════════════════════════════════════════════════════
# Installs required packages and downloads the Hˢ pipeline.
# Safe to re-run — skips what's already installed.

import subprocess, sys, os, importlib

def install(pkg):
    """Install a package if not already available."""
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg} already installed")
    except ImportError:
        print(f"  ↓ Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"  ✓ {pkg} installed")

print("Hˢ Standards Edition — Checking dependencies...")
print()

# Core dependencies
install("numpy")
install("matplotlib")
install("pandas")

print()
print("All dependencies ready.")
print()

# ── Locate or download pipeline ──
PIPELINE_DIR = None
REPO_URL = "https://raw.githubusercontent.com/PeterHiggins19/higgins-decomposition/main"

# Check local paths
for candidate in [
    'tools/pipeline',
    'pipeline',
    '../tools/pipeline',
    '../../tools/pipeline',
    os.path.join(os.path.dirname(os.path.abspath('__file__')), 'pipeline'),
    os.path.join(os.path.dirname(os.path.abspath('__file__')), 'tools', 'pipeline'),
]:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, 'higgins_decomposition_12step.py')):
        PIPELINE_DIR = os.path.abspath(candidate)
        break

if PIPELINE_DIR:
    print(f"Pipeline found locally: {PIPELINE_DIR}")
else:
    # Download from GitHub
    print("Pipeline not found locally — downloading from GitHub...")
    PIPELINE_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'hs_pipeline')
    os.makedirs(PIPELINE_DIR, exist_ok=True)
    os.makedirs(os.path.join(PIPELINE_DIR, 'locales'), exist_ok=True)

    import urllib.request

    pipeline_files = [
        'higgins_decomposition_12step.py',
        'higgins_transcendental_pretest.py',
        'hs_codes.py',
        'hs_reporter.py',
        'hs_fingerprint.py',
        'hs_amalgamation.py',
        'hs_sensitivity.py',
        'hs_metrology.py',
        'hs_ingest.py',
        'hs_hepdata.py',
        'hs_testgen.py',
        'hs_audit.py',
        'hs_controller.py',
    ]
    locale_files = ['en.json', 'zh.json', 'hi.json', 'pt.json', 'it.json']

    for f in pipeline_files:
        target = os.path.join(PIPELINE_DIR, f)
        if not os.path.exists(target):
            url = f"{REPO_URL}/tools/pipeline/{f}"
            try:
                urllib.request.urlretrieve(url, target)
                print(f"  ✓ {f}")
            except Exception as e:
                print(f"  ✗ {f}: {e}")

    for f in locale_files:
        target = os.path.join(PIPELINE_DIR, 'locales', f)
        if not os.path.exists(target):
            url = f"{REPO_URL}/tools/pipeline/locales/{f}"
            try:
                urllib.request.urlretrieve(url, target)
                print(f"  ✓ locales/{f}")
            except Exception as e:
                print(f"  ✗ locales/{f}: {e}")

    print(f"Pipeline downloaded to: {PIPELINE_DIR}")

# Add to path and import
sys.path.insert(0, PIPELINE_DIR)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown, FileLink

from higgins_decomposition_12step import HigginsDecomposition, NumpyEncoder
from hs_codes import generate_codes, CODE_DICTIONARY, STRUCTURAL_MODES

try:
    from hs_reporter import report
    HAS_REPORTER = True
except:
    HAS_REPORTER = False

try:
    from hs_fingerprint import extract_fingerprint
    HAS_FINGERPRINT = True
except:
    HAS_FINGERPRINT = False

try:
    from hs_amalgamation import SubcompositionalRecursion
    HAS_AMALGAMATION = True
except:
    HAS_AMALGAMATION = False

try:
    from hs_sensitivity import ComponentPowerMapper
    HAS_POWER_MAP = True
except:
    HAS_POWER_MAP = False

print()
print(f"Pipeline:       {PIPELINE_DIR}")
print(f"Codes:          {len(CODE_DICTIONARY)} diagnostic codes, {len(STRUCTURAL_MODES)} structural modes")
print(f"Reporter:       {'✓' if HAS_REPORTER else '✗'}")
print(f"Fingerprint:    {'✓' if HAS_FINGERPRINT else '✗'}")
print(f"Amalgamation:   {'✓' if HAS_AMALGAMATION else '✗'}")
print(f"Power Mapper:   {'✓' if HAS_POWER_MAP else '✗'}")
print()
print("━" * 50)
print("  Hˢ Standards Edition — READY")
print("━" * 50)

---

## Reference Standards

Three built-in datasets for immediate testing. Run any of these before loading your own data to verify the pipeline is working correctly.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2: REFERENCE STANDARDS — Built-in test datasets
# ════════════════════════════════════════════════════════════

# ── Standard 1: Nuclear SEMF (flagship result) ──
# Weizsäcker semi-empirical mass formula, Z=1→92
# Expected: NATURAL, δ = 5.87×10⁻⁶ at 1/(π^e), Z=38

def make_semf():
    aV, aS, aC, aA, aP = 15.56, 17.23, 0.7, 23.285, 12.0
    rows = []
    for Z in range(1, 93):
        A = round(2.5 * Z)
        vol = aV * A
        surf = aS * A**(2/3)
        coul = aC * Z * (Z - 1) / A**(1/3)
        asym = aA * (A - 2*Z)**2 / A
        pair = 0
        if Z % 2 == 0 and (A - Z) % 2 == 0: pair = aP / A**0.5
        elif Z % 2 == 1 and (A - Z) % 2 == 1: pair = -aP / A**0.5
        p1, p2, p3 = abs(vol), abs(surf) + abs(coul), abs(asym) + abs(pair)
        rows.append([p1, p2, p3])
    return np.array(rows), ['Volume', 'Surf+Coul', 'Sym+Pair']

# ── Standard 2: Simple 3-carrier composition ──
# Clean synthetic data — known NATURAL classification

def make_simple():
    np.random.seed(42)
    base = np.array([0.5, 0.3, 0.2])
    data = base + np.random.normal(0, 0.02, (30, 3))
    data = np.abs(data)
    data = data / data.sum(axis=1, keepdims=True)
    return data, ['A', 'B', 'C']

# ── Standard 3: Planck 2018 cosmic energy budget ──
# 5-carrier composition across redshift z=0→3400

def make_cosmic():
    H0 = 67.36; Om = 0.3153; Ob = 0.0493; Or = 9.14e-5
    On = 0.6847; Orad = Or * (1 + 0.2271 * 3.044 / 3)
    Ocdm = Om - Ob
    z_vals = np.concatenate([
        np.linspace(0, 10, 20),
        np.linspace(10, 100, 15),
        np.linspace(100, 1100, 15),
        np.linspace(1100, 3400, 10)
    ])
    rows = []
    for z in z_vals:
        a = 1 / (1 + z)
        rho_de = On
        rho_cdm = Ocdm / a**3
        rho_b = Ob / a**3
        rho_g = Or / a**4
        rho_nu = (Orad - Or) / a**4
        total = rho_de + rho_cdm + rho_b + rho_g + rho_nu
        rows.append([rho_de/total, rho_cdm/total, rho_b/total,
                     rho_g/total, rho_nu/total])
    return np.array(rows), ['DarkEnergy', 'CDM', 'Baryons', 'Photons', 'Neutrinos']

STANDARDS = {
    'SEMF': ('Nuclear SEMF (Z=1→92)', 'NUCLEAR', make_semf),
    'Simple': ('Simple 3-Carrier', 'TEST', make_simple),
    'Cosmic': ('Planck 2018 Cosmic Budget', 'COSMOLOGY', make_cosmic),
}

print("Available reference standards:")
for key, (name, domain, _) in STANDARDS.items():
    print(f"  {key:8s} — {name} [{domain}]")
print()
print("Select a standard below, or skip to Cell 4 to load your own CSV.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3: SELECT YOUR DATA SOURCE
# ════════════════════════════════════════════════════════════
# Option A: Use a reference standard (change the key below)
# Option B: Load a CSV file (set CSV_FILE path)
# Option C: Skip to Cell 4 for interactive upload

# ═══ CONFIGURE HERE ═══
MODE = "standard"          # "standard" or "csv"
STANDARD_KEY = "SEMF"      # "SEMF", "Simple", or "Cosmic"
CSV_FILE = None             # e.g., "mydata.csv" or "/path/to/data.csv"
SYSTEM_NAME = None          # Override name (auto-detected if None)
DOMAIN = None               # Override domain (auto-detected if None)
REPORT_LANG = "en"          # en, zh, hi, pt, it
# ═══════════════════════

if MODE == "csv" and CSV_FILE and os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    data = df[numeric_cols].values
    carriers = numeric_cols
    exp_name = SYSTEM_NAME or os.path.splitext(os.path.basename(CSV_FILE))[0].replace('_', ' ').title()
    exp_domain = DOMAIN or "USER_DATA"
    print(f"Loaded CSV: {CSV_FILE}")
elif MODE == "standard" and STANDARD_KEY in STANDARDS:
    name, domain, maker = STANDARDS[STANDARD_KEY]
    data, carriers = maker()
    exp_name = SYSTEM_NAME or name
    exp_domain = DOMAIN or domain
    print(f"Using reference standard: {exp_name}")
else:
    # Default to SEMF
    name, domain, maker = STANDARDS['SEMF']
    data, carriers = maker()
    exp_name = name
    exp_domain = domain
    print(f"Defaulting to: {exp_name}")

N, D = data.shape

# Remove any zero or negative rows
valid = np.all(data > 0, axis=1)
if not np.all(valid):
    removed = N - valid.sum()
    data = data[valid]
    N = data.shape[0]
    print(f"  ⚠ Removed {removed} rows with zero/negative values")

print(f"N={N} observations, D={D} carriers")
print(f"Carriers: {', '.join(carriers)}")
print()

# Data preview
df_preview = pd.DataFrame(data[:8], columns=carriers)
try:
    display(df_preview.style.set_caption(f"First 8 of {N} observations — {exp_name}").format('{:.6g}'))
except Exception:
    display(df_preview)

---

## Interactive CSV Upload

If you have a CSV file on your computer, run the cell below. It will create a file upload widget — click "Upload", select your CSV, and the data loads automatically.

**CSV format:** Each column is a carrier (component), each row is an observation. Numeric columns only — text columns are ignored.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4: INTERACTIVE CSV UPLOAD (optional)
# ════════════════════════════════════════════════════════════
# Run this cell, click Upload, select your CSV.
# After upload, re-run cells below to analyse.

try:
    import ipywidgets as widgets
    from IPython.display import clear_output

    upload_widget = widgets.FileUpload(accept='.csv,.tsv,.txt', multiple=False,
                                        description='Upload CSV')
    output_area = widgets.Output()

    def on_upload(change):
        with output_area:
            clear_output()
            global data, carriers, exp_name, exp_domain, N, D
            uploaded = upload_widget.value
            if uploaded:
                # Handle both old and new ipywidgets API
                if isinstance(uploaded, dict):
                    fname = list(uploaded.keys())[0]
                    content = uploaded[fname]['content']
                elif isinstance(uploaded, tuple) and len(uploaded) > 0:
                    fname = uploaded[0].name if hasattr(uploaded[0], 'name') else 'uploaded.csv'
                    content = uploaded[0].content if hasattr(uploaded[0], 'content') else uploaded[0]
                else:
                    print("Upload format not recognised")
                    return

                import io
                if isinstance(content, bytes):
                    text = content.decode('utf-8')
                else:
                    text = str(content)

                df_up = pd.read_csv(io.StringIO(text))
                numeric_cols = df_up.select_dtypes(include=[np.number]).columns.tolist()
                if len(numeric_cols) < 2:
                    print("⚠ Need at least 2 numeric columns. Found:", df_up.columns.tolist())
                    return

                data = df_up[numeric_cols].values
                # Remove zero/negative rows
                valid = np.all(data > 0, axis=1)
                if not np.all(valid):
                    print(f"  ⚠ Removed {(~valid).sum()} rows with zero/negative values")
                    data = data[valid]

                carriers = numeric_cols
                N, D = data.shape
                exp_name = os.path.splitext(fname)[0].replace('_', ' ').title()
                exp_domain = "USER_DATA"

                print(f"✓ Loaded: {fname}")
                print(f"  N={N} observations, D={D} carriers")
                print(f"  Carriers: {', '.join(carriers)}")
                print()
                display(pd.DataFrame(data[:8], columns=carriers).style
                        .set_caption(f"First 8 of {N} — {exp_name}").format('{:.6g}'))
                print()
                print("Now run the cells below (Shift+Enter) to analyse.")

    upload_widget.observe(on_upload, names='value')
    display(widgets.VBox([
        widgets.HTML("<div style='font-size:14px;color:#555;margin-bottom:8px'>"
                     "Click <b>Upload</b> to load a CSV file:</div>"),
        upload_widget,
        output_area
    ]))

except ImportError:
    print("ipywidgets not available. Use Cell 3 with CSV_FILE instead.")
    print("Install with: pip install ipywidgets")

---

## Full Pipeline Analysis

Run this cell to execute the complete Hˢ 12-step pipeline on your data.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5: RUN FULL Hˢ PIPELINE
# ════════════════════════════════════════════════════════════

print(f"Running Hˢ pipeline on: {exp_name} ({exp_domain})")
print(f"Data: {N} observations × {D} carriers")
print()

hd = HigginsDecomposition(
    experiment_id="HS-STD",
    name=exp_name,
    domain=exp_domain,
    carriers=carriers,
    data_source_type="STANDARDS_EDITION",
    data_source_description=f"Hˢ Standards Edition — {exp_name}"
)
hd.load_data(data)
result = hd.run_full_extended()

# Generate codes
codes = generate_codes(result)
sm_codes = [c for c in codes if c['code'].startswith('SM-')]
pipeline_codes = [c for c in codes if not c['code'].startswith('SM-')]

# Classification
classification = 'NATURAL' if any(c['code'] == 'S8-NAT-INF' for c in codes) \
    else 'INVESTIGATE' if any(c['code'] == 'S8-INV-WRN' for c in codes) else 'FLAG'
cls_color = {'NATURAL': '#3FB950', 'INVESTIGATE': '#F0883E', 'FLAG': '#F85149'}[classification]

# Key metrics
s = result['steps']
hvld_shape = s.get('step6_pll_shape', '?')
hvld_r2 = s.get('step6_pll_R2', 0)
sq = s.get('step7_squeeze_closest')
nearest_const = sq.get('constant', '—') if sq else '—'
nearest_delta = sq.get('delta', 0) if sq else 0
eitt = s.get('step8_eitt_invariance', {})
eitt_pass = all(v.get('pass', False) for v in eitt.values()) if eitt else False

# Transfer entropy
ext = result.get('extended', {})
xc = ext.get('conditional', {})
te = xc.get('transfer_entropy', {})
te_text = ""
if te:
    pairs = te.get('top_pairs', [])
    if pairs:
        te_text = " | ".join([f"{p.get('from','?')}→{p.get('to','?')}" for p in pairs[:3]])

# Summary card
display(HTML(f"""
<div style='background:#161B22;color:#E6EDF3;padding:20px;border-radius:12px;border:3px solid {cls_color};
     font-family:Consolas,monospace;margin:12px 0;max-width:700px'>
  <div style='font-size:24px;color:#FFD700;font-weight:bold;margin-bottom:4px'>Hˢ Pipeline Results</div>
  <div style='font-size:11px;color:#8B949E;margin-bottom:16px'>Higgins Decomposition on the Simplex — Standards Edition</div>
  <table style='color:#E6EDF3;font-size:14px;border-collapse:collapse;width:100%'>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E;width:160px'>System</td><td style='font-weight:bold;font-size:16px'>{exp_name}</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>Domain</td><td>{exp_domain}</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>N × D</td><td>{N} × {D}</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>Classification</td><td style='color:{cls_color};font-weight:bold;font-size:20px'>{classification}</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>HVLD shape</td><td>{hvld_shape} (R² = {hvld_r2:.4f})</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>Nearest constant</td><td style='color:#FFD700'>{nearest_const} (δ = {nearest_delta:.2e})</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>EITT entropy</td><td style='color:{"#3FB950" if eitt_pass else "#F0883E"}'>{"PASS — invariant" if eitt_pass else "FAIL — variant"}</td></tr>
    <tr><td style='padding:4px 20px 4px 0;color:#8B949E'>Codes emitted</td><td>{len(codes)} ({len(pipeline_codes)} pipeline + {len(sm_codes)} structural modes)</td></tr>
    {"<tr><td style='padding:4px 20px 4px 0;color:#8B949E'>Transfer entropy</td><td style='color:#39D2C0'>" + te_text + "</td></tr>" if te_text else ""}
  </table>
  <div style='margin-top:12px;padding-top:10px;border-top:1px solid #30363D;font-size:11px;color:#8B949E'>
    Hˢ = R ∘ M ∘ E ∘ C ∘ T ∘ V ∘ S — The instrument reads. The expert decides. The loop stays open.
  </div>
</div>
"""))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6: DASHBOARD VISUALISATION
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 10), facecolor='#0D1117')
for ax in axes.flat:
    ax.set_facecolor('#161B22')
    ax.tick_params(colors='#8B949E', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#30363D')

colors_bar = ['#FFD700', '#39D2C0', '#F0883E', '#BC8CFF', '#58A6FF', '#F85149', '#3FB950']

# ── Panel 1: σ²_A Variance Trajectory ──
ax1 = axes[0, 0]
sig2 = s.get('step5_sigma2_trajectory', [])
if sig2:
    ax1.plot(sig2, color='#58A6FF', linewidth=1.5)
    if sq:
        const_val = sq.get('constant_value', 0)
        if const_val and const_val < max(sig2) * 1.2:
            ax1.axhline(y=const_val, color='#FFD700', linestyle='--', linewidth=1, alpha=0.7)
            ax1.text(len(sig2)*0.02, const_val*1.05, f"{nearest_const}", color='#FFD700', fontsize=8)
ax1.set_title('σ²_A Variance Trajectory', color='#FFD700', fontsize=10, fontweight='bold')
ax1.set_xlabel('Observation', color='#8B949E', fontsize=8)
ax1.set_ylabel('σ²_A', color='#8B949E', fontsize=8)

# ── Panel 2: Composition bars ──
ax2 = axes[0, 1]
closed = np.array(s.get('step3_closed_data', data / data.sum(axis=1, keepdims=True)))
indices = [0, N//2, N-1]
bar_width = 0.25
for bi, idx in enumerate(indices):
    row = closed[min(idx, len(closed)-1)]
    positions = np.arange(D) + bi * bar_width
    for j in range(D):
        ax2.bar(positions[j], row[j], bar_width * 0.9, color=colors_bar[j % len(colors_bar)], alpha=0.7)
ax2.set_xticks(np.arange(D) + bar_width)
ax2.set_xticklabels([c[:8] for c in carriers], fontsize=7, color='#8B949E', rotation=30)
ax2.set_title('Compositions (first / mid / last)', color='#FFD700', fontsize=10, fontweight='bold')
ax2.set_ylabel('Proportion', color='#8B949E', fontsize=8)

# ── Panel 3: Ternary or CLR ──
ax3 = axes[0, 2]
clr_data = np.array(s.get('step4_clr_data', []))
if len(clr_data) > 0 and D == 3:
    def to_ternary(row):
        ss = row[0] + row[1] + row[2]
        x = 0.5 * (2 * row[1] + row[2]) / ss
        y = (np.sqrt(3) / 2) * row[2] / ss
        return x, y
    pts = np.array([to_ternary(r) for r in closed])
    ax3.scatter(pts[:, 0], pts[:, 1], c=plt.cm.viridis(np.linspace(0, 1, N)), s=20, zorder=3)
    ax3.plot(pts[:, 0], pts[:, 1], color='#58A6FF', alpha=0.3, linewidth=0.8)
    tri = np.array([[0,0],[1,0],[0.5,np.sqrt(3)/2],[0,0]])
    ax3.plot(tri[:, 0], tri[:, 1], color='#30363D', linewidth=0.5)
    ax3.set_title('Ternary Projection', color='#FFD700', fontsize=10, fontweight='bold')
elif len(clr_data) > 0:
    ax3.scatter(clr_data[:, 0], clr_data[:, 1], c=np.arange(len(clr_data)), cmap='viridis', s=20)
    ax3.axhline(y=0, color='#30363D', linewidth=0.5)
    ax3.axvline(x=0, color='#30363D', linewidth=0.5)
    ax3.set_xlabel(f'CLR({carriers[0][:8]})', color='#8B949E', fontsize=8)
    ax3.set_ylabel(f'CLR({carriers[1][:8]})', color='#8B949E', fontsize=8)
    ax3.set_title('CLR Projection', color='#FFD700', fontsize=10, fontweight='bold')
ax3.set_aspect('equal')

# ── Panel 4: Shannon Entropy ──
ax4 = axes[1, 0]
entropy = s.get('step8_entropy_trajectory', [])
if entropy:
    ax4.plot(entropy, color='#3FB950', linewidth=1.5)
    mean_h = np.mean(entropy)
    ax4.axhline(y=mean_h, color='#F85149', linestyle='--', linewidth=1, alpha=0.6)
    ax4.text(len(entropy)*0.02, mean_h + 0.02, f'H̄ = {mean_h:.4f}', color='#F85149', fontsize=8)
ax4.set_title(f'Shannon Entropy (EITT: {"PASS" if eitt_pass else "FAIL"})',
              color='#FFD700', fontsize=10, fontweight='bold')
ax4.set_xlabel('Observation', color='#8B949E', fontsize=8)
ax4.set_ylabel('H (normalised)', color='#8B949E', fontsize=8)

# ── Panel 5: Complex Plane ──
ax5 = axes[1, 1]
if len(clr_data) > 0 and clr_data.shape[1] >= 2:
    re_vals = clr_data[:, 0] - np.mean(clr_data[:, 0])
    im_vals = clr_data[:, 1] - np.mean(clr_data[:, 1])
    ax5.scatter(re_vals, im_vals, c=plt.cm.plasma(np.linspace(0, 1, len(clr_data))), s=20, zorder=3)
    ax5.plot(re_vals, im_vals, color='#BC8CFF', alpha=0.2, linewidth=0.8)
    ax5.axhline(y=0, color='#30363D', linewidth=0.5)
    ax5.axvline(x=0, color='#30363D', linewidth=0.5)
ax5.set_title('Complex Plane (CLR Phase)', color='#FFD700', fontsize=10, fontweight='bold')
ax5.set_xlabel('Re(CLR)', color='#8B949E', fontsize=8)
ax5.set_ylabel('Im(CLR)', color='#8B949E', fontsize=8)
ax5.set_aspect('equal')

# ── Panel 6: Per-carrier contribution ──
ax6 = axes[1, 2]
pcc = ext.get('universal', {}).get('per_carrier_contribution', {})
contributions = pcc.get('contributions', {})
if contributions:
    carr_names = list(contributions.keys())
    carr_vals = [contributions[c] for c in carr_names]
    ax6.barh(range(len(carr_names)), carr_vals,
             color=[colors_bar[i % len(colors_bar)] for i in range(len(carr_names))], alpha=0.8)
    ax6.set_yticks(range(len(carr_names)))
    ax6.set_yticklabels([c[:12] for c in carr_names], fontsize=8, color='#E6EDF3')
    for i, v in enumerate(carr_vals):
        ax6.text(v + 0.5, i, f'{v:.1f}%', color='#E6EDF3', fontsize=8, va='center')
ax6.set_title('Per-Carrier Variance Contribution', color='#FFD700', fontsize=10, fontweight='bold')
ax6.set_xlabel('% of total variance', color='#8B949E', fontsize=8)

plt.suptitle(f'Hˢ — {exp_name} ({exp_domain}) — {classification}',
             color='#FFD700', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Hs_dashboard.png', dpi=120, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f"Dashboard saved → Hs_dashboard.png")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7: DIAGNOSTIC CODES & STRUCTURAL MODES
# ════════════════════════════════════════════════════════════

level_colors = {'INF': '#3FB950', 'WRN': '#F0883E', 'ERR': '#F85149',
                'DIS': '#58A6FF', 'CAL': '#BC8CFF'}

# Pipeline codes table
rows_html = []
for c in pipeline_codes:
    level = c['code'].split('-')[-1]
    color = level_colors.get(level, '#8B949E')
    rows_html.append(
        f"<tr><td style='font-family:Consolas;color:{color};padding:3px 10px;white-space:nowrap'>"
        f"[{c['code']}]</td>"
        f"<td style='color:#E6EDF3;padding:3px 10px'>{c['short']}</td></tr>"
    )

display(HTML(f"""
<div style='background:#161B22;padding:16px;border-radius:8px;margin:8px 0'>
<div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:10px'>
  Diagnostic Codes ({len(pipeline_codes)} emitted)
</div>
<table style='border-collapse:collapse;width:100%'>{''.join(rows_html)}</table>
</div>
"""))

# Structural modes
if sm_codes:
    sm_html = ""
    for c in sm_codes:
        sm_html += f"""
        <div style='background:#1c1c2e;padding:10px 14px;margin:6px 0;border-left:3px solid #FFD700;border-radius:4px'>
          <div style='font-family:Consolas;color:#FFD700;font-size:13px;font-weight:bold'>
            [{c['code']}] {c['short']}
          </div>
          <div style='color:#E6EDF3;font-size:12px;margin-top:4px;line-height:1.5'>
            {c['verbose']}
          </div>
        </div>
        """

    display(HTML(f"""
    <div style='background:#161B22;padding:16px;border-radius:8px;border:2px solid #FFD700;margin:8px 0'>
      <div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:12px'>
        Structural Modes — Investigation Prompts ({len(sm_codes)} detected)
      </div>
      <div style='color:#8B949E;font-size:11px;margin-bottom:10px'>
        Second-order diagnoses suggesting specific investigation paths.
        The tool asks — the expert answers.
      </div>
      {sm_html}
    </div>
    """))
else:
    print("No structural modes fired.")

---

## Advanced Analysis

The cells below run optional advanced analyses: amalgamation stability, component power mapping, and fingerprint generation.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8: AMALGAMATION STABILITY TEST
# ════════════════════════════════════════════════════════════
# Tests whether the classification survives when carriers are merged.
# 100% stability = the structure is robust to subcomposition.

if HAS_AMALGAMATION and D >= 3:
    print(f"Running amalgamation stability test ({D} carriers)...")
    print()

    engine = SubcompositionalRecursion(verbose=False)
    amal_results = engine.run(exp_name, carriers, data)

    # Count classification preservation across amalgamation levels
    total_schemes = 0
    preserved = 0
    for level in amal_results.get('levels', []):
        for scheme in level.get('schemes', []):
            total_schemes += 1
            if scheme.get('classification') == classification:
                preserved += 1

    stability = (preserved / total_schemes * 100) if total_schemes > 0 else 0
    stab_color = '#3FB950' if stability >= 95 else '#F0883E' if stability >= 80 else '#F85149'

    display(HTML(f"""
    <div style='background:#161B22;padding:16px;border-radius:8px;border:2px solid {stab_color};margin:8px 0'>
      <div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:8px'>
        Amalgamation Stability
      </div>
      <div style='font-size:28px;color:{stab_color};font-weight:bold;font-family:Consolas'>
        {preserved}/{total_schemes} schemes ({stability:.1f}%)
      </div>
      <div style='color:#8B949E;font-size:12px;margin-top:4px'>
        Classification "{classification}" preserved under all valid carrier merges.
      </div>
    </div>
    """))
elif D < 3:
    print("Amalgamation requires D ≥ 3 carriers.")
else:
    print("Amalgamation engine not available.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9: COMPONENT POWER MAP
# ════════════════════════════════════════════════════════════
# Identifies which carrier drives the most variance.

if HAS_POWER_MAP:
    print(f"Running Component Power Mapper...")
    print()

    mapper = ComponentPowerMapper(data, carriers, domain=exp_domain)
    power_map = mapper.run_full_analysis()

    # Display power scores
    scores = power_map.get('power_scores', {}).get('carriers', {})
    if scores:
        rows_pm = []
        for carrier_name, score_data in sorted(scores.items(), key=lambda x: x[1].get('power_score', 0), reverse=True):
            total = score_data.get('power_score', 0)
            leverage = score_data.get('norm_leverage', 0)
            phase = score_data.get('norm_criticality', 0)
            bar_w = min(total * 3, 100)
            rows_pm.append(
                f"<tr><td style='padding:4px 12px;color:#E6EDF3;font-weight:bold'>{carrier_name}</td>"
                f"<td style='padding:4px 12px'>"
                f"<div style='background:#30363D;border-radius:3px;width:200px;height:16px'>"
                f"<div style='background:#FFD700;border-radius:3px;width:{bar_w*2}px;height:16px'></div>"
                f"</div></td>"
                f"<td style='padding:4px 12px;color:#FFD700;font-family:Consolas'>{total:.2f}</td>"
                f"<td style='padding:4px 12px;color:#8B949E;font-family:Consolas'>{leverage:.2f}</td>"
                f"<td style='padding:4px 12px;color:#8B949E;font-family:Consolas'>{phase:.2f}</td></tr>"
            )

        display(HTML(f"""
        <div style='background:#161B22;padding:16px;border-radius:8px;margin:8px 0'>
          <div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:10px'>
            Component Power Map
          </div>
          <table style='border-collapse:collapse'>
            <tr style='color:#8B949E;font-size:11px'>
              <th style='padding:4px 12px;text-align:left'>Carrier</th>
              <th style='padding:4px 12px;text-align:left'>Power</th>
              <th style='padding:4px 12px'>Total</th>
              <th style='padding:4px 12px'>Leverage</th>
              <th style='padding:4px 12px'>Phase</th>
            </tr>
            {''.join(rows_pm)}
          </table>
        </div>
        """))
else:
    print("Power Mapper not available.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10: COMPOSITIONAL FINGERPRINT
# ════════════════════════════════════════════════════════════
# Generates a unique 7-dimensional geometric identity for this system.

if HAS_FINGERPRINT:
    fp = extract_fingerprint(result, name=exp_name, domain=exp_domain)

    fp_hash = fp.get('fingerprint', {}).get('hash', '—')
    fp_vec = fp.get('fingerprint', {}).get('vector', {})

    dims_html = ""
    dim_names = list(fp_vec.keys()) if fp_vec else ['shape', 'r2_band', 'classification']
    for dn in dim_names:
        val = fp_vec.get(dn, 0)
        if isinstance(val, (int, float)):
            bar_w = min(abs(val) * 100, 200)
        else:
            bar_w = 50  # nominal bar for string values
            val = str(val) if val else '—'
        dims_html += (
            f"<tr><td style='padding:2px 10px;color:#8B949E;font-size:12px'>{dn}</td>"
            f"<td style='padding:2px 10px'>"
            f"<div style='background:#30363D;border-radius:2px;width:200px;height:10px;display:inline-block'>"
            f"<div style='background:#58A6FF;border-radius:2px;width:{bar_w}px;height:10px'></div>"
            f"</div></td>"
            f"<td style='padding:2px 10px;color:#E6EDF3;font-family:Consolas;font-size:11px'>{val if isinstance(val, str) else f'{val:.6f}'}</td></tr>"
        )

    display(HTML(f"""
    <div style='background:#161B22;padding:16px;border-radius:8px;margin:8px 0'>
      <div style='color:#FFD700;font-weight:bold;font-size:16px;margin-bottom:4px'>
        Compositional Fingerprint
      </div>
      <div style='font-family:Consolas;color:#58A6FF;font-size:14px;margin-bottom:12px'>
        {fp_hash}
      </div>
      <table style='border-collapse:collapse'>{dims_html}</table>
    </div>
    """))
else:
    print("Fingerprint generator not available.")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11: FULL DIAGNOSTIC REPORT
# ════════════════════════════════════════════════════════════

if HAS_REPORTER:
    rpt = report(codes, lang=REPORT_LANG, result=result)
    display(HTML(f"""
    <details style='background:#161B22;padding:12px;border-radius:8px;margin:8px 0'>
      <summary style='color:#FFD700;font-weight:bold;cursor:pointer;font-size:14px'>
        Full Diagnostic Report ({REPORT_LANG.upper()}) — click to expand
      </summary>
      <pre style='color:#E6EDF3;font-family:Consolas;font-size:11px;margin-top:8px;
           white-space:pre-wrap'>{rpt}</pre>
    </details>
    """))
else:
    print("Reporter not available — showing raw codes:")
    for c in codes:
        print(f"  [{c['code']}] {c['short']}")

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12: SAVE RESULTS
# ════════════════════════════════════════════════════════════
# Saves everything to JSON files you can download or share.

import json

safe_name = exp_name.replace(' ', '_').replace('/', '_')

# Save pipeline results
result_path = f"Hs_{safe_name}_results.json"
with open(result_path, 'w') as f:
    json.dump(result, f, indent=2, cls=NumpyEncoder)

# Save diagnostic codes
codes_path = f"Hs_{safe_name}_codes.json"
with open(codes_path, 'w') as f:
    json.dump(codes, f, indent=2, default=str)

# Save report
if HAS_REPORTER:
    report_path = f"Hs_{safe_name}_report_{REPORT_LANG}.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report(codes, lang=REPORT_LANG, result=result))
else:
    report_path = None

# Save fingerprint
if HAS_FINGERPRINT:
    fp_path = f"Hs_{safe_name}_fingerprint.json"
    with open(fp_path, 'w') as f:
        json.dump(fp, f, indent=2, default=str)
else:
    fp_path = None

print("━" * 50)
print("  FILES SAVED")
print("━" * 50)
print(f"  Results:     {result_path}")
print(f"  Codes:       {codes_path}")
if report_path:
    print(f"  Report:      {report_path}")
if fp_path:
    print(f"  Fingerprint: {fp_path}")
print(f"  Dashboard:   Hs_dashboard.png")
print("━" * 50)
print()
print("The instrument reads. The expert decides. The loop stays open.")
print()
print("Peter Higgins — Rogue Wave Audio")
print("github.com/PeterHiggins19/higgins-decomposition")

---

## Pipeline Reference

| Step | Operator | Operation | What it produces |
|------|----------|-----------|------------------|
| S1-S3 | **S** | Simplex closure | System identity + closed N×D matrix |
| S4-S5 | **V** | Variance trajectory | CLR coordinates + σ²_A shape |
| S6 | **T** | CLR Transform | Per-carrier deviation mapping |
| S7 | **C** | Classification | Transcendental squeeze → NATURAL / INVESTIGATE / FLAG |
| S8 | **E** | Entropy test | EITT geometric-mean decimation invariance |
| S9-S10 | **M** | Mode synthesis | Structural modes from code conjunction |
| S11-S12 | **R** | Report | Assembly, fingerprint, SHA-256 verification |

## What to do with each classification

| Result | Meaning | Next step |
|--------|---------|-----------|
| **NATURAL** | System has inherent compositional structure anchored to a transcendental constant | Read the structural modes — they tell you what the structure IS |
| **INVESTIGATE** | Structure detected but near a decision boundary | Try alternate carrier decomposition, check sample size, examine outliers |
| **FLAG** | No structure detected, or structure is artificial | Check data quality, consider whether the carriers form a natural composition |

---

## Need Help?

| Resource | Link |
|----------|------|
| Repository | [github.com/PeterHiggins19/higgins-decomposition](https://github.com/PeterHiggins19/higgins-decomposition) |
| Learning Path | `docs/Hs_Learning_Path.md` in the repository |
| Full Reference | `docs/reference/Higgins_Decomposition_Reference_v3.0.docx` |
| CoDaWork 2026 Abstract | `papers/codawork2026/CoDaWork2026_Abstract_Higgins.pdf` |

---

*Hˢ Standards Edition v1.0 — April 2026*
*Peter Higgins — Rogue Wave Audio — Markham, Ontario, Canada*
*The instrument reads. The expert decides. The loop stays open.*